In [ ]:
CLASSIFIER_FOLDER = "./ClassifierModels/"

In [ ]:
import os

os.environ["DINOV3_LOCATION"] = r"C:\Users\Jan Magne\OneDrive - Akershus fylkeskommune\dinov3"

In [ ]:
# Sjekke om miljøvariabelen er satt riktig

print("DINOV3_LOCATION:", os.getenv("DINOV3_LOCATION"))

In [ ]:
#importere nødvendige biblioteker

import os
import pickle
import torch
from PIL import Image
from scipy import signal
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt


In [ ]:
# sett DINOv3 location til miljøvariabelen

DINOV3_LOCATION = os.getenv("DINOV3_LOCATION")

if DINOV3_LOCATION is None:
    raise ValueError("DINOV3_LOCATION environment variabel er ikke satt. Se tidligere steg.")

print("DINOv3 location set to:", DINOV3_LOCATION)

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

In [ ]:
# Vi starer med denne, da denne er minst og krever mindre ressurser
MODEL_DINOV3_VITS = "dinov3_vits16"

# Andre modeller du kan prøve
# Husk at du må laste ned "weights" for den modellen du ønsker å bruke
MODEL_DINOV3_VITSP = "dinov3_vits16plus"
MODEL_DINOV3_VITB = "dinov3_vitb16"
MODEL_DINOV3_VITL = "dinov3_vitl16"
MODEL_DINOV3_VITHP = "dinov3_vith16plus"
MODEL_DINOV3_VIT7B = "dinov3_vit7b16"

# we take DINOv3 ViT-S (since we have the pretrained weights for this model)
MODEL_NAME = MODEL_DINOV3_VITSP
 # Her kan du velge en annen modell 

# Load model without pretrained weights to avoid web download
model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model=MODEL_NAME,
    source="local",
    pretrained=True 
)

# Set model to evaluation mode and move to GPU
model.eval()
model.cuda()


# Test the model with a small dummy input to see if it produces valid output
print("\nTesting model with dummy input...")
dummy_input = torch.randn(1, 3, 224, 224).cuda()
with torch.no_grad():
    try:
        test_output = model(dummy_input)
        print(f"Model test successful. Output shape: {test_output.shape}")
        print(f"Output has NaN: {torch.isnan(test_output).any()}")
        print(f"Output range: {test_output.min():.4f} to {test_output.max():.4f}")
    except Exception as e:
        print(f"Model test failed: {e}")
        print("There might be an issue with the model or checkpoint loading.")

In [ ]:
# Konstanter for patch-størrelse og bilde-størrelse

PATCH_SIZE = 16    # Hver patch er 16×16 piksler
IMAGE_SIZE = 768   # Standard høyde vi skalerer til (768÷16 = 48 patches høyt)

In [ ]:
# Denne funksjonen bruker vi videre for å endre størrelse på maskene slik at de passer med patch-størrelsen
def resize_transform(mask_image: Image, image_size: int = IMAGE_SIZE, patch_size: int = PATCH_SIZE) -> torch.Tensor:
    w, h = mask_image.size                              # Original størrelse
    h_patches = int(image_size / patch_size)            # Antall patches vertikalt (768÷16=48)
    w_patches = int((w * image_size) / (h * patch_size)) # Antall patches horisontalt
    return TF.to_tensor(TF.resize(mask_image, (h_patches * patch_size, w_patches * patch_size)))

In [ ]:

with open(CLASSIFIER_FOLDER + "fg_classifier_{}.pkl".format(MODEL_NAME), 'rb') as f:
    clf = pickle.load(f)

# Self-attention lagene i DINOv3 ViT modellene
MODEL_TO_NUM_LAYERS = {
    MODEL_DINOV3_VITS: 12,
    MODEL_DINOV3_VITSP: 12,
    MODEL_DINOV3_VITB: 12,
    MODEL_DINOV3_VITL: 24,
    MODEL_DINOV3_VITHP: 32,
    MODEL_DINOV3_VIT7B: 40,
}

n_layers = MODEL_TO_NUM_LAYERS[MODEL_NAME]

IMAGENET_MEAN = (0.485, 0.456, 0.406) # RGB mean for ImageNet
IMAGENET_STD = (0.229, 0.224, 0.225) # RGB std for ImageNet

In [ ]:
folder = r"./data/test_images/"

for filename in sorted(os.listdir(folder)):
        if filename.endswith(('.png', '.jpg', '.jpeg')):

            test_image = Image.open(os.path.join(folder, filename))
            test_image_resized = resize_transform(test_image)
            test_image_normalized = TF.normalize(test_image_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)

            with torch.inference_mode():
                with torch.autocast(device_type='cuda', dtype=torch.float32):
                    feats = model.get_intermediate_layers(test_image_normalized.unsqueeze(0).cuda(), n=range(n_layers), reshape=True, norm=True)
                    x = feats[-1].squeeze().detach().cpu()
                    dim = x.shape[0]
                    x = x.view(dim, -1).permute(1, 0)

            h_patches, w_patches = [int(d / PATCH_SIZE) for d in test_image_resized.shape[1:]]

            fg_score = clf.predict_proba(x)[:, 1].reshape(h_patches, w_patches)
            fg_score_mf = torch.from_numpy(signal.medfilt2d(fg_score, kernel_size=3))

            plt.figure(figsize=(9, 3), dpi=300)
            plt.subplot(1, 3, 1)
            plt.axis('off')
            plt.imshow(test_image_resized.permute(1, 2, 0))
            plt.title('input image')
            plt.subplot(1, 3, 2)
            plt.axis('off')
            plt.imshow(fg_score)
            plt.title('foreground score')
            plt.subplot(1, 3, 3)
            plt.axis('off')
            plt.imshow(fg_score_mf)
            plt.title('+ median filter')
            plt.show()